# 03 — Customer Analytics: Repeat Purchases, Cohorts, and RFM

**Business purpose.** Describe observed purchase frequency and activity by cohort among customers with identifiable CustomerIDs. This is not a population-wide retention estimate and not customer lifetime value.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from src.config import MARTS_DIR, DEFAULT_CLEANED

if not DEFAULT_CLEANED.exists():
    raise FileNotFoundError("Run `python -m src.clean` first.")
rfm = pd.read_csv(MARTS_DIR / "customer_rfm.csv", dtype={"customer_id":"string"})
cohort = pd.read_csv(MARTS_DIR / "cohort_retention.csv", parse_dates=["cohort_month", "activity_month"])
print("Identified customers in RFM mart:", f"{len(rfm):,}")
display(rfm.head())

## Purchase frequency and RFM segments

Frequency is the count of distinct invoices with at least one qualifying positive line. RFM labels are deterministic behavioral groupings and should not be described as verified customer intentions.

In [ ]:
if not rfm.empty:
    display(rfm["segment"].value_counts().rename_axis("segment").to_frame("customers"))
    fig, ax = plt.subplots(figsize=(9, 4))
    freq = rfm["frequency_invoices"].value_counts().sort_index()
    ax.bar(freq.index.astype(str), freq.values)
    ax.set_title("Identified customers by eligible purchase-invoice count")
    ax.set_xlabel("Eligible purchase invoices")
    ax.set_ylabel("Customers")
    plt.show()
else:
    print("No qualifying identified customers were present; review audit and source schema.")

## Cohort activity

`cohort_index = 1` is the first eligible purchase month, and its retention/activity rate equals 100% by construction. Recent cohorts have fewer observed follow-up months; never compare their incomplete tail as a full retention curve.

In [ ]:
display(cohort.head(20))
if not cohort.empty:
    matrix = cohort.pivot(index="cohort_month", columns="cohort_index", values="retention_rate")
    fig, ax = plt.subplots(figsize=(12, 6))
    image = ax.imshow(matrix.values, aspect="auto", vmin=0, vmax=1)
    ax.set_title("Observed cohort activity rate by elapsed month")
    ax.set_xlabel("Cohort index (months since first purchase)")
    ax.set_ylabel("First-purchase cohort month")
    ax.set_xticks(range(len(matrix.columns)), labels=matrix.columns)
    ax.set_yticks(range(len(matrix.index)), labels=[x.strftime("%Y-%m") for x in matrix.index])
    fig.colorbar(image, ax=ax, label="Observed activity rate")
    plt.tight_layout()
    plt.show()

## Sensitivity and interpretation checkpoint

Compare the distribution of customer frequency and RFM monetary values with the audit’s missing CustomerID share. Then describe one way the result could change under a different monetary definition or observation date. Record that alternative in your own words before moving to management recommendations.